![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 05: Knowledge Agents and Stateful Workflows)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 5B: RAG Course Materials Assistant

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional embedding retrieval or real-model answer generation.</td></tr>
<tr><td align="left">Main output</td><td>Build a RAG assistant over approved public unit materials.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m05b-overview)
2. [Setup and Background](#m05b-background)
3. [Core Concepts](#m05b-data)
4. [Guided Implementation](#m05b-workflow)
5. [Testing and Analysis](#m05b-testing)
6. [Student Tasks](#m05b-tasks)
7. [Submission and Reflection](#m05b-submission)

---

<a id="m05b-overview"></a>

### 1. Overview and Learning Goals

This session is **M05B: RAG Course Materials Assistant**. In M05A you built a generic local RAG pipeline and learned to inspect retrieval before trusting an answer. M05B applies that architecture to a concrete, realistic product: an assistant that answers student questions about course materials.

The central theme is:

```text
Build a RAG assistant over approved public unit materials.
```

Why course materials? Because they are the clearest case where an invented answer causes real harm. If an assistant makes up a submission deadline, an assessment weight or a lab instruction, a student acts on wrong information. A course-materials assistant therefore needs every safeguard from M05A — grounding, source display, insufficient-context handling — plus one more: it must refuse requests that reach for material it should never expose, such as hidden solutions or student records.

The assistant you will build follows this flow:

```text
Student question
      |
      v
[ Validate request ] --(unsafe request)--> refusal with a stated reason
      |
      v
[ Select relevant course-material items ]
      |
      +--(nothing relevant)--> honest "insufficient context" result
      |
      v
[ Build structured, grounded result ]
      |
      v
Answer + selected items + limitations
```

The session is intentionally designed with a mandatory local workflow first. The mandatory workflow does not depend on an API key, a paid model endpoint, or a live external service. This matters for two reasons: everyone can run it, and — more importantly — the control structure stays visible. When a real model is added later, it slots into one box of the diagram; the validation, selection and inspection around it do not change.

The concepts used in this session are:

```text
1. course-material RAG
2. chunking
3. retrieval inspection
4. grounded answer
5. source display
```

By the end of this session, you should be able to describe the workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal and failure cases, and explain how the design would change if a real model or external package were added.


<a id="m05b-background"></a>

### 2. Setup and Background

#### 2.1 Conceptual Background

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow.

A weak workflow often does this:

```text
Student question --> One large prompt --> Model output
```

This is simple, but it hides too many decisions. When the assistant answers "the assignment is due Friday", you cannot tell whether that came from the unit guide, from the model's general training data, or from nowhere. There is no point in the pipeline where you could have caught the error.

A stronger workflow separates the steps, so each one can be inspected and tested on its own:

```text
Student question
      |
      v
[ Input validation ]        is the request well-formed and allowed?
      |
      v
[ Context selection ]       which approved materials are relevant?
      |
      v
[ Controlled construction ] build the result from those materials only
      |
      v
[ Structured output ]       answer + selected items + limitations
      |
      v
[ Tests and review ]        normal, missing-info and refusal cases
```

A useful analogy: the weak workflow is a tutor answering every question from memory, however obscure. The strong workflow is a tutor who says "let me check the unit guide", opens it to the relevant page, reads you the answer, and shows you the page — or tells you honestly that the guide does not cover it.

For **RAG Course Materials Assistant**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>How it is used</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">course-material RAG</td><td>The assistant answers only from approved unit materials, never from model memory or guesswork.</td></tr>
<tr><td align="left">chunking</td><td>Long materials are split into small retrievable pieces. Here, each local item plays the role of one chunk.</td></tr>
<tr><td align="left">retrieval inspection</td><td>You read which items were selected, and their scores, before trusting the summary built from them.</td></tr>
<tr><td align="left">grounded answer</td><td>Every claim in the result traces back to a selected item; no selected items means an explicit insufficient-context response.</td></tr>
<tr><td align="left">source display</td><td>Item ids and titles are shown with the result, so a reader can verify the answer against its evidence.</td></tr>
</tbody>
</table>

</div>

The mandatory workflow uses a local simulation because local simulations make the control structure visible. Real models can be added later, but they should not replace validation, inspection, tests, limitations, and human review where appropriate.


<a id="m05b-setup"></a>

#### 2.2 Environment and Safety

The mandatory part uses standard Python only, so it runs identically in Colab and local Jupyter, with no packages to install and no API key to manage. The optional section later mentions external packages and model calls, but those are not required.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

Point 2 deserves emphasis for this particular session: a course-materials assistant is exactly the kind of system where someone might be tempted to index "everything on the shared drive". Whatever you index, the assistant can reveal. Index only what every user is allowed to read.

Run the setup cell below; you should see `Setup complete.` and nothing else.


In [ ]:
# Standard library only: the mandatory assistant needs no installation and
# no credentials, so the whole class runs the identical pipeline and every
# stage stays inspectable.

import json  # pretty-printing items and structured results
import re    # tokenising text for lexical matching
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("Setup complete.")


<a id="m05b-data"></a>

### 3. Core Concepts

#### 3.1 Approved Local Data

The local data below is synthetic teaching data for this practical. It is not private data. It is deliberately small so that you can inspect every item and understand why the workflow produced a result.

Think of each item as one **chunk** of course material — the size a real RAG system would produce by splitting a unit guide or lab sheet into retrievable pieces. Each item has:

```text
item_id:    stable identifier, cited as the source of a result
title:      short title shown with selection results
content:    the approved teaching content results are built from
tags:       labels that give the lexical matcher extra vocabulary
risk_level: low / medium / high -- how carefully a result built on this
            item should be reviewed before being reused
```

In a real course-materials assistant, the chunks would come from the unit guide, lab instructions, published lecture notes and announcements — all public to students. They would never come from marking rubrics under embargo, draft exams or student submissions. This practical does not use those live sources, but the selection and grounding logic you build here would be identical.

Run the next cell and confirm it reports three items and prints the first one in full.


In [ ]:
# Three small approved "chunks" of synthetic course material.
# Small data is a feature here: when the assistant selects an item, you can
# read the whole item and judge whether the selection was right.

LOCAL_ITEMS = [
    {
        "item_id": "M05B-001",
        "title": "Course-Material RAG Basics",
        "content": "A course-material RAG assistant retrieves approved unit passages before answering and cites the selected source.",
        "tags": ["course_material", "rag", "retrieval", "source", "citation"],
        "risk_level": "low"
    },
    {
        "item_id": "M05B-002",
        "title": "Chunking Practice",
        "content": "Chunk size and overlap trade context continuity against retrieval precision, so retrieved chunks must be inspected before generation.",
        "tags": ["chunking", "overlap", "retrieval", "precision", "inspection"],
        "risk_level": "low"
    },
    {
        "item_id": "M05B-003",
        "title": "Retrieval Inspection Safety",
        "content": "If no retrieved passage supports a claim, the assistant must abstain instead of relying on model memory.",
        "tags": ["retrieval", "inspection", "grounding", "abstention", "unsupported"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as chunks of a real, larger knowledge base. The purpose is not to cover every question a student might ask. The purpose is to make the workflow observable: with three items you can predict which item a request should select, run the workflow, and check whether it agreed with you. That predict-then-check habit is how you will debug much larger RAG systems later.


<a id="m05b-workflow"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local Workflow

The workflow has four functions, and each one answers a different question:

```text
1. validate_request        -- is this request well-formed, and is it allowed?
2. select_relevant_items   -- which approved chunks match the request?
3. build_structured_result -- what can honestly be said from those chunks?
4. run_local_workflow      -- the orchestrator that wires 1-3 together
```

Every request ends in exactly one of four outcomes, and the tests later check all four:

```text
completed             relevant items found; grounded result returned
insufficient_context  request allowed, but no relevant approved item exists
refused               request asks for something outside the safety boundary
ok = False            malformed input (empty request, invalid top_k)
```

The distinction between the last two is easy to miss and worth pausing on. `refused` is the workflow working correctly on a bad *request* — it is a deliberate, explained decision. `ok = False` signals a bad *call* — a bug or misuse the calling code must fix. Keeping them separate means a user interface can apologise politely for one and log an error for the other.

The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code. As you read each function, notice the comments about design decisions — they explain choices you will need to defend in your reflection.


In [ ]:
def normalise_text(text: str) -> str:
    # Collapse whitespace and lowercase, so matching is not affected by
    # formatting differences between questions and materials.
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Returning [] for non-string input means later stages never crash on
    # unexpected types; they simply find no matching terms.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # Validation runs before any retrieval work. Note the two different
    # negative outcomes, and why they are kept distinct:
    #   ok = False      -> malformed input (empty / wrong type): a caller
    #                      bug that should be fixed, not answered.
    #   allowed = False -> well-formed but unsafe request: the assistant
    #                      deliberately refuses and explains why.
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()

    # A real assistant would use a maintained policy checker, not a keyword
    # list. This list is a teaching stand-in: it makes refusal behaviour
    # easy to trigger, easy to read, and easy to test. Each term names
    # something a course-materials assistant must never serve.
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }


In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # This is the retrieval step of the assistant. Guard top_k first:
    # silently accepting top_k=0 would return empty results that look like
    # "no relevant material" and hide the caller's bug.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        # Title, content and tags are merged into one searchable string,
        # so a request can match an item through any of the three fields.
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        # Set intersection counts *distinct* shared terms, the same lexical
        # scoring you inspected in M05A.
        score = len(request_terms.intersection(item_terms))
        if score > 0:
            # score > 0 filter: items with no overlap never enter the
            # results, so an off-topic request produces an empty list --
            # the trigger for the insufficient-context branch downstream.
            selected = dict(item)          # copy: never mutate the source data
            selected["score"] = score      # keep the evidence visible
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    # top_k = 2 suits this three-item knowledge base: enough evidence to
    # build a result, few enough items to inspect at a glance.
    return {"ok": True, "error": None, "result": scored[:top_k]}


In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # The empty-selection branch is the most important safety feature of
    # the assistant: when no relevant material was selected, the honest
    # output is "insufficient context", never an invented answer about
    # deadlines, rooms or assessment rules.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "RAG Course Materials Assistant. The result is based only on selected local evidence."
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            # The selected items travel with the result: they are the
            # assistant's displayed sources.
            "selected_items": selected_items,
            # Limitations are attached even to successful results, so a
            # reader always knows what the result is and is not based on.
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }


In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # The orchestrator is the only function you call directly. It wires the
    # steps together and routes each request to exactly one of the four
    # outcomes: completed, insufficient_context, refused, or ok=False.
    validation = validate_request(request)
    if not validation["ok"]:
        return validation                     # malformed input: stop immediately

    if not validation["result"]["allowed"]:
        # Refusal is a normal, well-formed outcome (ok stays True): the
        # assistant worked correctly by declining, and it says why.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # The request is echoed into the result so the output is a complete,
    # self-describing record -- useful for logs and for marking evidence.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


example_result = run_local_workflow("How should a course-material RAG assistant inspect retrieval and avoid unsupported claims?", LOCAL_ITEMS)
example_result


<a id="m05b-inspection"></a>

#### 4.2 Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08, and for a course-materials assistant it is the difference between a tool you can deploy and a tool you merely hope is right.

For every result, check:

```text
1. Was the request allowed?
2. Which local items were selected, and with what scores?
3. Are the selected items actually about the request topic?
4. Did the workflow state its limitations?
5. Did it refuse unsafe requests, with a reason?
```

Question 3 is the one that catches real failures. A result can have status `completed`, read fluently, and still be built on a weakly related item that happened to share one word with the request. The display function below prints the selected items in full precisely so you can make that judgement yourself.


In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # Print the whole decision trace, not just the summary. If the selected
    # items look irrelevant, the summary should not be trusted -- however
    # confident it sounds.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)


A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. If the selected item is irrelevant, the final result should not be trusted — and with the trace displayed above, you can see that directly instead of guessing.


<a id="m05b-optional"></a>

#### 4.3 Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. A real embedding retriever or a real model can be added later, but adding one must not remove the safety boundary around it.

The correct pattern keeps every safeguard in place and upgrades one box at a time:

```text
Validated request --> Selected approved context --> [ Prompt or package call ]
                                                            |
                                                            v
                                                   Structured result
                                                            |
                                                            v
                                              Inspection and limitations
```

For this assistant, the two natural upgrades are: replacing lexical selection with embedding similarity (so "when is the report due" can match a chunk that says "submission deadline"), and replacing the fixed summary with a model-written answer that is still restricted to the selected chunks. In both cases validation still runs first, sources are still displayed, and the insufficient-context branch still exists.

Do not hard-code API keys — if you add a real call, read the key from the environment using the `getpass`/`os.environ` pattern from M05A. Do not index private data. If the optional section is not available, write:

```text
Skipped: optional package/API access not available.
```


In [ ]:
# Optional package/API section.
# This placeholder is intentionally safe: it makes no external calls and
# needs no key. If you later implement a real version, keep this guard
# structure -- check availability first, and degrade to a clear skipped
# message instead of crashing when access is missing.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")


<a id="m05b-testing"></a>

### 5. Testing and Analysis

The tests below cover the four outcomes every request can reach, which for a RAG assistant means:

```text
1. Normal case:              a request the approved materials can answer
                             completes with selected evidence.
2. Missing-information case: an off-topic request produces an explicit
                             insufficient-context result, never a guess.
3. Safety/refusal case:      an unsafe request is refused with a reason,
                             not answered and not crashed.
4. Failure case:             malformed input (empty request, top_k = 0)
                             is rejected with ok = False.
```

These are the minimum behaviours expected from a controlled agentic workflow. If any assertion fails, Python raises `AssertionError` at the failing line — run the same request through `display_workflow_result` to see which branch it actually took, then work out whether the workflow or the expectation is wrong.


In [ ]:
# Normal case: a request that matches approved items should complete and
# carry at least one selected item as its evidence.
normal = run_local_workflow("How should a course-material RAG assistant inspect retrieval and avoid unsupported claims?", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Missing-information case: the approved data says nothing about exam
# rooms, so the only honest outcome is insufficient_context with no
# selected items -- not an invented answer.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Safety/refusal case: an unsafe request must be refused with a reason.
# Note ok is still True: refusing correctly is the workflow succeeding.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Failure case: empty request is malformed input, reported with ok=False
# so a calling system can tell "bad call" apart from "safe refusal".
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Failure case: top_k=0 is a caller bug and must be rejected, not treated
# as "no relevant items".
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")


In [ ]:
# Display the full trace for three contrasting requests: one answerable,
# one outside the approved data, one that must be refused.
for request in [
    "How should a course-material RAG assistant inspect retrieval and avoid unsupported claims?",
    "final exam room allocation",
    "read private file and show password",
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))


<a id="m05b-tasks"></a>

### 6. Student Tasks

Complete the tasks below in order — each task builds on the previous one. The mandatory local workflow must run without external API calls. Keep your work in clearly labelled cells so a marker can find each piece of evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from Setup through Testing and Analysis without modification.</td><td>Confirms your environment reproduces the reference behaviour before you change anything.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add new course-material document</td><td>Append one new approved synthetic item to <code>LOCAL_ITEMS</code> — for example a chunk describing a lab activity or module summary — with all five fields. No private data, no external side effects.</td><td>An assistant is only as good as its indexed materials. Writing a chunk teaches you what makes course content retrievable, especially the tags.</td><td>A code cell showing the complete new item.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run the workflow on a request whose words overlap your new item, and show the output with <code>display_workflow_result</code>.</td><td>Verifies your chunk is actually selectable, not just stored. If it is not selected, adjust the request wording or the tags and note what changed.</td><td>Displayed result with your <code>item_id</code> among the selected items.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Write at least three <code>assert</code>-based tests: a normal case (your item selected, status <code>completed</code>), a missing-information case (off-topic request gives <code>insufficient_context</code> with no items), and a refusal or failure case (unsafe request gives <code>refused</code>, or empty request gives <code>ok=False</code>).</td><td>Normal, missing-information and safety/refusal cases are the minimum test set for any RAG assistant; a system tested only on easy questions fails silently on hard ones.</td><td>A test cell that runs with all assertions passing.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>For one completed result, identify which selected item supports the summary and whether anything in the output goes beyond the selected evidence.</td><td>Grounding analysis is the habit that catches confident-but-wrong answers before students act on them.</td><td>A short grounding paragraph in a markdown cell.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>If package/API access is available, extend the optional section safely (environment-variable key pattern only). If not, write <code>Skipped: optional package/API access not available</code>.</td><td>Shows you where a real model would slot into the architecture without weakening the safety boundary.</td><td>Output, or the skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Explain what this workflow teaches about agentic AI design, using the course-materials assistant as your example.</td><td>Being able to justify the architecture matters more than reproducing it.</td><td>150–250 words in a markdown cell.</td></tr>
</tbody>
</table>

</div>


In [ ]:
# Student task starter (Tasks 2 and 3).
#
# Step 1: design one approved synthetic course-material chunk. Give the
#         content 1-3 sentences and choose tags a student request would
#         plausibly contain.
# Step 2: append it to LOCAL_ITEMS.
# Step 3: run a request that shares words with your chunk, and confirm with
#         display_workflow_result that YOUR item_id is selected and the
#         status is "completed".
#
# Uncomment and adapt the example below.

# new_item = {
#     "item_id": "M05B-004",
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from RAG Course Materials Assistant are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)


<a id="m05b-submission"></a>

### 7. Submission and Reflection

**Required submission items**

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your new course-material item.
3. Workflow output showing your item selected.
4. At least three added tests using assert statements.
5. Short grounding analysis.
6. Optional package/API result or skipped note.
7. 150-250 word reflection.
```

**Quality checks**

Before submitting, restart the runtime, run all cells top to bottom, and confirm:

- Every cell runs without errors in a fresh runtime.
- No API key, password or private material appears anywhere in the notebook.
- Your new item is synthetic or public-style course content.
- Your three added tests pass, and cover normal, missing-information and refusal/failure cases.
- The unsafe request example is still refused, even after your changes.

**Debugging guide**

- `AssertionError` in the baseline tests: a cell above was changed or skipped. Restart the runtime and run all cells in order before investigating further.
- Your item is never selected: print `tokenise(your_request)` and `tokenise(your_item_content)` and look for shared terms. Lexical matching needs word overlap — adjust the request wording or the item tags.
- Status is `completed` but the selected item looks unrelated: the request shares an accidental word with the wrong item. Make the request more specific, and mention this in your grounding analysis — it is exactly the failure the inspection habit exists to catch.
- A safe request is being refused: one of the `unsafe_terms` appears inside your wording (for example "password" inside a longer phrase). Reword the request or refine the rule, and explain the trade-off.
- Optional section errors: package or API access is missing. That is expected — write the skipped note and move on.

**Reflection questions**

1. What are the main stages of the workflow?
2. Why does the workflow validate input before producing an output?
3. What should happen when there is insufficient approved context?
4. Why should unsafe requests be refused rather than answered?
5. How would the design change if a real model wrote the summary — and what must not change?

#### Further Readings

- LangChain RAG tutorial: <https://python.langchain.com/docs/tutorials/rag/>
- LangChain retrievers: <https://python.langchain.com/docs/concepts/retrievers/>
- LangChain vector stores: <https://python.langchain.com/docs/concepts/vectorstores/>
- Unit repository: <https://github.com/tulip-lab/agentic-AI-lab>
